# 04 · The chart

One horizontal stacked bar per health publisher, showing the share of resource links
that are hard-dead, uncertain or OK, with the link count printed on every bar.

**Reading it.** I order each bar *hard-dead, then uncertain, then OK*, left to right:
- The end of the red segment is the confirmed-dead share, the floor.
- The end of the grey segment is what the broken share would be if every uncertain link were dead, the ceiling.

The two dashed lines are the NDL's *national* rates as quoted (about 30% before the cleanup,
about 7% after). They're there for scale. They aren't my figures, and their scope (the whole catalogue,
tested June 2026) differs from mine (six health publishers, tested on the snapshot date).

**Colour.** Red is hard-dead. Neutral grey with hatching is uncertain: it has no hue on purpose,
because it means "unknown". Blue is OK. I validated these with the dataviz palette checker, which
passes on CVD separation, normal-vision separation and contrast. The grey fails only the chroma
floor, deliberately, and the hatching and legend carry it.

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve()
ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
from hclr import config


In [2]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

t = pd.read_parquet(config.GOLD / "by_publisher_final.parquet")
snapshot = pd.read_parquet(config.SILVER / "resources.parquet")["snapshot_utc"].iloc[0]
MIN_N = 30  # below this I show the bar but flag it: a 100% bar of 2 links means nothing

short = {
    "NHS Digital (merged into NHS England, 2023)": "NHS Digital\n(predecessor; now NHS England)",
    "Department of Health and Social Care": "Department of Health\nand Social Care",
    "NHS England": "NHS England",
    "NHS Business Services Authority": "NHS Business Services\nAuthority",
    "Public Health England (closed 2021; to UKHSA/OHID)": "Public Health England\n(predecessor; closed 2021)",
    "UK Health Security Agency": "UK Health Security\nAgency",
}
t["small"] = t["links"] < MIN_N
# Sorted by hard-dead share; ties broken by the uncertain ceiling, then size. Small-n bars always go last.
t = t.sort_values(["small", "hard_dead_pct", "broken_ceiling_pct", "links"], ascending=[True, False, False, False])
t[["links", "hard_dead", "uncertain", "ok", "hard_dead_pct", "uncertain_pct", "ok_pct"]].round(1)

final_class,links,hard_dead,uncertain,ok,hard_dead_pct,uncertain_pct,ok_pct
publisher,,,,,,,
"NHS Digital (merged into NHS England, 2023)",2489,96,1733,660,3.9,69.6,26.5
Department of Health and Social Care,1160,10,762,388,0.9,65.7,33.4
NHS England,439,0,20,419,0.0,4.6,95.4
NHS Business Services Authority,198,0,0,198,0.0,0.0,100.0
Public Health England (closed 2021; to UKHSA/OHID),17,0,0,17,0.0,0.0,100.0
UK Health Security Agency,2,0,0,2,0.0,0.0,100.0


In [3]:
SURFACE, INK, INK2, MUTED, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#8a8984", "#e6e5e0"
C_DEAD, C_UNC, C_OK = "#d03b3b", "#8c8a82", "#2a78d6"

plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 10, "axes.edgecolor": GRID,
                     "text.color": INK, "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK})
fig, ax = plt.subplots(figsize=(10, 6.8), dpi=200)
fig.patch.set_facecolor(SURFACE); ax.set_facecolor(SURFACE)

rows = list(t.index)[::-1]          # matplotlib draws bottom-up
y = range(len(rows))
H = 0.56
for yi, pub in zip(y, rows):
    r = t.loc[pub]
    alpha = 0.45 if r["small"] else 1.0
    left = 0
    for val, colour, hatch in [(r["hard_dead_pct"], C_DEAD, None), (r["uncertain_pct"], C_UNC, "////"), (r["ok_pct"], C_OK, None)]:
        if val > 0:
            ax.barh(yi, val, left=left, height=H, color=colour, alpha=alpha, hatch=hatch,
                    edgecolor=SURFACE, linewidth=2)   # 2px surface gap between segments
        left += val
    # Shares printed inside a segment only where they fit comfortably.
    if r["uncertain_pct"] >= 9:
        ax.text(r["hard_dead_pct"] + r["uncertain_pct"] / 2, yi, f"{r['uncertain_pct']:.0f}% uncertain",
                ha="center", va="center", color="white", fontsize=8.5, fontweight="bold", zorder=7,
                bbox=dict(boxstyle="round,pad=0.25", fc=C_UNC, ec="none", alpha=alpha))
    if r["ok_pct"] >= 12:
        ax.text(100 - r["ok_pct"] / 2, yi, f"{r['ok_pct']:.0f}% OK", ha="center", va="center",
                color="white", fontsize=8.5, fontweight="bold", alpha=1 if not r["small"] else 0.85)
    # The hard-dead share is the story, so it always gets a label, just past the red segment.
    dead_label = (f"{r['hard_dead_pct']:.1f}% hard-dead ({int(r['hard_dead'])} links)" if r["hard_dead"] else "0 hard-dead")
    ax.annotate(dead_label, xy=(0, yi + H / 2), xytext=(1, 3), textcoords="offset points",
                ha="left", va="bottom", fontsize=8, color=C_DEAD if r["hard_dead"] else MUTED,
                fontweight="bold" if r["hard_dead"] else "normal", zorder=7,
                bbox=dict(boxstyle="square,pad=0.15", fc=SURFACE, ec="none"))  # sits above the dashed lines
    # Link count on every bar, outside the 100% end, so no bar implies more than it holds.
    n_txt = f"n = {int(r['links']):,} links" + ("\ntoo few to judge" if r["small"] else "")
    ax.text(101.5, yi, n_txt, ha="left", va="center", fontsize=8.5, color=INK2 if not r["small"] else MUTED)

# National reference lines: quoted, not mine.
for x, lab in [(7, "National ~7%\nafter cleanup"), (30, "National ~30%\nbefore cleanup")]:
    ax.axvline(x, color=INK, lw=1.1, ls=(0, (4, 3)), zorder=5)
    ax.text(x, len(rows) - 0.25, lab, ha="center", va="bottom", fontsize=8, color=INK, zorder=7,
            bbox=dict(boxstyle="square,pad=0.2", fc=SURFACE, ec="none"))

ax.set_yticks(list(y), [short[p] for p in rows])
ax.set_xlim(0, 100); ax.set_ylim(-0.6, len(rows) - 0.1 + 0.55)
ax.set_xticks([0, 20, 40, 60, 80, 100], ["0%", "20%", "40%", "60%", "80%", "100%"])
ax.tick_params(axis="y", length=0); ax.tick_params(axis="x", length=0)
ax.xaxis.grid(True, color=GRID, lw=0.8); ax.set_axisbelow(True)
for side in ["top", "right", "left"]:
    ax.spines[side].set_visible(False)
ax.set_xlabel("Share of resource links on data.gov.uk")

fig.suptitle("Few health links are confirmed dead. Most of the legacy ones can't be checked at all",
             x=0.012, y=0.985, ha="left", fontsize=13, fontweight="bold")
fig.text(0.012, 0.935, f"Share of resource links by status, health and social care publishers on data.gov.uk, checked {snapshot[:10]}",
         ha="left", fontsize=9.5, color=INK2)
handles = [Patch(fc=C_DEAD, ec=SURFACE, label="Hard-dead (404, 410, DNS/connection failure)"),
           Patch(fc=C_UNC, ec=SURFACE, hatch="////", label="Uncertain (403, timeout, WAF/CAPTCHA, other)"),
           Patch(fc=C_OK, ec=SURFACE, label="OK (2xx)"),
           Line2D([0], [0], color=INK, lw=1.1, ls=(0, (4, 3)), label="National rate, as quoted by the NDL")]
fig.legend(handles=handles, loc="lower left", bbox_to_anchor=(0.005, 0.075), ncol=2, frameon=False, fontsize=8.5)
fig.text(0.012, 0.012,
         "Source: data.gov.uk CKAN API snapshot and my own link check (HEAD-first, 2 s per host, no retries; browser re-check for 403s, timeouts and challenges).\n"
         "National lines: National Data Library, 'cleaning up data.gov.uk' (12 Nov 2026), quoted and not independently verified. Different scope and date from mine: for scale only.\n"
         "NHS Digital and Public Health England are predecessor bodies, shown separately and not merged into their successors.",
         fontsize=7, color=MUTED, ha="left", va="bottom")
fig.subplots_adjust(left=0.255, right=0.86, top=0.85, bottom=0.235)
config.OUTPUTS.mkdir(exist_ok=True)
fig.savefig(config.OUTPUTS / "health_linkrot_by_publisher.png", facecolor=SURFACE)
fig.savefig(config.OUTPUTS / "health_linkrot_by_publisher.svg", facecolor=SURFACE)
print("saved outputs/health_linkrot_by_publisher.png and .svg")

saved outputs/health_linkrot_by_publisher.png and .svg


## The table behind the chart

The chart is a static image for Medium, so the numbers live here and in `data/gold/by_publisher_final.csv`.

In [4]:
tbl = t[["links", "hard_dead", "uncertain", "ok", "hard_dead_pct", "hard_dead_ci_lo", "hard_dead_ci_hi",
         "uncertain_pct", "ok_pct", "broken_floor_pct", "broken_ceiling_pct"]].round(1)
tbl.to_csv(config.OUTPUTS / "chart_table.csv"); tbl

final_class,links,hard_dead,uncertain,ok,hard_dead_pct,hard_dead_ci_lo,hard_dead_ci_hi,uncertain_pct,ok_pct,broken_floor_pct,broken_ceiling_pct
publisher,,,,,,,,,,,
"NHS Digital (merged into NHS England, 2023)",2489,96,1733,660,3.9,3.2,4.7,69.6,26.5,3.9,73.5
Department of Health and Social Care,1160,10,762,388,0.9,0.5,1.6,65.7,33.4,0.9,66.6
NHS England,439,0,20,419,0.0,0.0,0.9,4.6,95.4,0.0,4.6
NHS Business Services Authority,198,0,0,198,0.0,0.0,1.9,0.0,100.0,0.0,0.0
Public Health England (closed 2021; to UKHSA/OHID),17,0,0,17,0.0,0.0,18.4,0.0,100.0,0.0,0.0
UK Health Security Agency,2,0,0,2,0.0,0.0,65.8,0.0,100.0,0.0,0.0
